# Ordered Logistic Regression Results for Adoption Predictors (FAIR²) Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is defined by a Croissant schema at:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install -U mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Version: {metadata.version}")
print("\nDescription:")
print(metadata.description)


### Basic Metadata
* **DOI**: 10.71728/senscience.y7m0-f273
* **License**: https://opendatacommons.org/licenses/by/1-0/
* **Keywords**: adoption predictors, climate adaptation, extension services, gender inclusion, indigenous knowledge
* **Timeframe**: 2021-11-16 to 2024-11-16
* **Location**: Samburu, Isiolo, Marsabit counties, Northern Kenya

## 2. Data Overview
Explore the record sets, fields, and columns defined in the Croissant schema.

We'll list all record sets and their fields, including their `@id`s for reference (as per Croissant best practices).

In [ ]:
# Show available record sets and their fields in the dataset
def print_record_sets_info(ds):
    print("Available Record Sets:")
    for rs in ds.record_sets:
        print(f"- RecordSet @id: {rs['@id']} (name: {rs.get('name', '[No Name]')})")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            print(f"    - Field @id: {f['@id']} (name: {f.get('name', '[No Name]')}, type: {f.get('dataType', '[unknown]')})")

print_record_sets_info(dataset)

# For reference, collect record set and field @ids in a list
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
if len(record_set_ids) == 0:
    print("No record sets defined in Croissant schema.")
else:
    print(f"\nRecordSet @ids: {record_set_ids}")

Now, let's print the first few records for each record set to understand their structure.

In [ ]:
# Show the first 2 records for each record set by @id (if available)
for rs_id in record_set_ids:
    print(f"\nFirst 2 records for RecordSet @id: {rs_id}")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            print(rec)
            if i == 1:
                break
    except Exception as e:
        print(f"  (Failed to load records: {e})")


## 3. Data Extraction
Extract data from record sets into DataFrames. Each entity is referenced strictly by its `@id`.

In [ ]:
# Get all available record set @ids
record_sets = record_set_ids  # already collected from above
dataframes = {}

# Attempt extraction for each record set
for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Loaded RecordSet {record_set} with shape {dataframes[record_set].shape}")
        print(f"Fields: {list(dataframes[record_set].columns)}")
    except Exception as e:
        print(f"Could not load records for {record_set}: {e}")

# For demonstration: pick first available DataFrame
if dataframes:
    chosen_record_set_id = list(dataframes.keys())[0]
    print(f"\nUsing RecordSet @id: {chosen_record_set_id} for further exploration.")
    display(dataframes[chosen_record_set_id].head())
else:
    print("No data loaded, data exploration will be limited.")

## 4. Exploratory Data Analysis (EDA)
Let's perform standard EDA: filtering, normalizing, and grouping on numeric fields. We reference all entities with their `@id`.

_For demonstration, we'll automatically attempt to select a numeric field and a group field if available._

In [ ]:
import numpy as np

if dataframes:
    df = dataframes[chosen_record_set_id]

    # Attempt to find a numeric field by heuristic (float or int columns)
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    for col in df.columns:
        if pd.api.types.is_categorical_dtype(df[col]) or df[col].dtype == object:
            group_field_id = col
            break
    if numeric_field_id:
        print(f"Numeric field detected for EDA: {numeric_field_id}")

        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {filtered_df.shape[0]} rows")

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Head of normalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group if grouping field available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No group field detected.")
    else:
        print("No numeric field detected in this record set.")
else:
    print("No data to analyze.")


## 5. Visualization
Visualize the distribution of the selected numeric field, and a grouped barplot if both numeric and group fields are available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(data=df, x=numeric_field_id, bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.barplot(data=df, x=group_field_id, y=numeric_field_id, estimator=np.mean, ci='sd')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.show()


## 6. Conclusion
In this notebook, we've demonstrated the use of the `mlcroissant` library to:
- Access dataset metadata and documentation programmatically.
- Enumerate all record sets, fields, and columns by their `@id`s as specified in the Croissant schema.
- Extract structured records into pandas DataFrames.
- Perform basic EDA such as filtering and normalizing by field `@id`.
- Visualize distributions for selected numeric fields and group summaries when possible.

This ensures a reproducible and schema-compliant approach to dataset exploration, particularly for complex or multi-table FAIR data packages.

You can now extend this workflow for statistical analysis, machine learning, or integrative multi-dataset studies as needed.